### Data Ingestion


LOading


In [2]:
import os
import json
import chromadb
from sentence_transformers import SentenceTransformer



c:\Users\Dell\postmate\MYpostman-APP\Backend\RAG-local\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader

target_path = os.path.join("..", "RAG-folder")
print(f"--- Loading from: {os.path.abspath(target_path)} ---")

loader_kwargs = {"encoding": "utf-8"}

# Load JSON and MD separately — brace expansion is unreliable on Windows
json_loader = DirectoryLoader(target_path, glob="**/*.json", loader_cls=TextLoader, loader_kwargs=loader_kwargs, show_progress=True)
md_loader   = DirectoryLoader(target_path, glob="**/*.md",   loader_cls=TextLoader, loader_kwargs=loader_kwargs, show_progress=True)

json_docs = json_loader.load()
md_docs   = md_loader.load()
documents = json_docs + md_docs

if not documents:
    print("⚠️  No files found.")
else:
    print(f"✅ Loaded {len(json_docs)} JSON + {len(md_docs)} MD = {len(documents)} total documents.")
    print(f"Preview: {documents[0].page_content[:100]}...")

--- Loading from: c:\Users\Dell\postmate\MYpostman-APP\Backend\RAG-local\RAG-folder ---


100%|██████████| 1/1 [00:00<00:00, 45.46it/s]

✅ Loaded 1 JSON + 1 MD = 2 total documents.
Preview: [
  {
    "id": 1,
    "category": "Technology Announcement",
    "tone": "Professional and concise"...


### Embeddings and Vector DB

In [4]:
from sklearn.metrics.pairwise import cosine_similarity
import uuid
from chromadb.config import Settings
import chromadb
from typing import List, Dict, Any
import os




In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np
import torch

class EmbeddingManager:
    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        """
        Constructor for EmbeddingManager.

        Args:
            model_name (str): SentenceTransformer model name
        """
        self.model_name = model_name
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model."""
        try:
            self.model = SentenceTransformer(self.model_name, device=self.device)
            print(f"Model '{self.model_name}' loaded successfully on {self.device}.")
            print(f"Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise

    def generate_embeddings(self, text: str | list[str]) -> np.ndarray:
        """
        Generate embeddings for a single text or a list of texts.

        Args:
            text (str | list[str]): Text or list of texts to embed.

        Returns:
            list[float]: Single embedding if text is a string.
            list[list[float]]: List of embeddings if text is a list.
            return array of embeddings as numpy array
        """
        if self.model is None:
            raise RuntimeError("Model is not loaded. Call _load_model() first.")
        embeddings = self.model.encode(text, normalize_embeddings=True)
        print(f"Generated embeddings for input: {text}")
        print(f"Embedding shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()
sample_text = "This is a sample text to generate embeddings."

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1514.67it/s]


Model 'sentence-transformers/all-MiniLM-L6-v2' loaded successfully on cpu.
Embedding dimension: 384


### Vector Store

In [6]:
class VectorStore:
    def __init__(self, collection_name: str = "my_collection_v2", persist_directory: str = "vector_store"):
        """
        Constructor for VectorStore.

        Args:
            collection_name (str): Name of the ChromaDB collection.
            persist_directory (str): Directory to persist the ChromaDB data.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
   
        self._initialize_vector_store()

    def _initialize_vector_store(self):                          # ✅ level 1
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection for storing document embeddings and metadata",
                          "hnsw:space": "cosine"} # <--- ADD THIS LINE
            )
            print(f"Vector store initialized with collection '{self.collection_name}' at '{self.persist_directory}'.")
            print(f"Collection metadata: {self.collection.metadata}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):   # ✅ level 1
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents and embeddings must be the same.")

        ids, metadatas, document_texts, embeddings_list = [], [], [], []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            document_texts.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:                                                     # ✅ outside loop
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=document_texts
            )
            print(f"Added {len(documents)} documents to the vector store.")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

    

In [7]:
vectorstore=VectorStore()
vectorstore

Vector store initialized with collection 'my_collection_v2' at 'vector_store'.
Collection metadata: {'description': 'Collection for storing document embeddings and metadata', 'hnsw:space': 'cosine'}


### Creating Data Chunks

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [9]:
chunk=split_documents(documents)
chunk

Split 2 documents into 18 chunks

Example chunk:
Content: [
  {
    "id": 1,
    "category": "Technology Announcement",
    "tone": "Professional and concise",
    "content": "Just deployed the latest Docker containers for the MYpostmate backend! If anyone i...
Metadata: {'source': '..\\RAG-folder\\top-posts-archive.json'}


[Document(metadata={'source': '..\\RAG-folder\\top-posts-archive.json'}, page_content='[\n  {\n    "id": 1,\n    "category": "Technology Announcement",\n    "tone": "Professional and concise",\n    "content": "Just deployed the latest Docker containers for the MYpostmate backend! If anyone is struggling with port mapping between Node.js and PostgreSQL, let\'s discuss it here. \\ud83d\\ude80 #TechTalk"\n  },\n  {\n    "id": 2,\n    "category": "Community Discussion",\n    "tone": "Casual and helpful",\n    "content": "Friendly reminder: your reputation score directly affects your dashboard visibility. Keep the discussions clean and help out your fellow devs! What\'s everyone working on this weekend?"\n  },\n  {\n    "id": 3,\n    "category": "Debugging Help",\n    "tone": "Technical and analytical",\n    "content": "Stuck on a weird React state rendering issue. The component remounts twice on initial load. I\'ve checked the useEffect dependencies. Any ideas? Snippet attached below."\n  

In [10]:
## convert chunks text into embeddings
texts=[doc.page_content for doc in chunk]
texts

['[\n  {\n    "id": 1,\n    "category": "Technology Announcement",\n    "tone": "Professional and concise",\n    "content": "Just deployed the latest Docker containers for the MYpostmate backend! If anyone is struggling with port mapping between Node.js and PostgreSQL, let\'s discuss it here. \\ud83d\\ude80 #TechTalk"\n  },\n  {\n    "id": 2,\n    "category": "Community Discussion",\n    "tone": "Casual and helpful",\n    "content": "Friendly reminder: your reputation score directly affects your dashboard visibility. Keep the discussions clean and help out your fellow devs! What\'s everyone working on this weekend?"\n  },\n  {\n    "id": 3,\n    "category": "Debugging Help",\n    "tone": "Technical and analytical",\n    "content": "Stuck on a weird React state rendering issue. The component remounts twice on initial load. I\'ve checked the useEffect dependencies. Any ideas? Snippet attached below."\n  },\n  {\n    "id": 4,\n    "category": "Tech Discussion",\n    "tone": "Academic and 

In [11]:

embeddings = embedding_manager.generate_embeddings(texts)
vectorstore.add_documents(chunk, embeddings)

Generated embeddings for input: ['[\n  {\n    "id": 1,\n    "category": "Technology Announcement",\n    "tone": "Professional and concise",\n    "content": "Just deployed the latest Docker containers for the MYpostmate backend! If anyone is struggling with port mapping between Node.js and PostgreSQL, let\'s discuss it here. \\ud83d\\ude80 #TechTalk"\n  },\n  {\n    "id": 2,\n    "category": "Community Discussion",\n    "tone": "Casual and helpful",\n    "content": "Friendly reminder: your reputation score directly affects your dashboard visibility. Keep the discussions clean and help out your fellow devs! What\'s everyone working on this weekend?"\n  },\n  {\n    "id": 3,\n    "category": "Debugging Help",\n    "tone": "Technical and analytical",\n    "content": "Stuck on a weird React state rendering issue. The component remounts twice on initial load. I\'ve checked the useEffect dependencies. Any ideas? Snippet attached below."\n  },\n  {\n    "id": 4,\n    "category": "Tech Discussi

### Retrival RAG pipeline fetch from vector DB

In [12]:
from typing import List, Dict, Any

class Retrieval:
    def __init__(self, vectorstore, embedding_manager):
        """ 
        Initializes the Retrieval class with a vector store and an embedding manager. 
        """
        self.vectorstore = vectorstore
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, score_threshold: float = 0.0, top_k: int = 5) -> List[Dict[str, Any]]:
        """ 
        Retrieves relevant documents based on a query.
        """
        print(f"Retrieving documents for query: '{query}' with score threshold: {score_threshold} and top_k: {top_k}")
        
        # Step 1: Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        print(f"Query embedding shape: {query_embedding.shape}")
        
        # Step 2: Retrieve relevant documents from the vector store
        try:
            results = self.vectorstore.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                # ✅ FIX 1: Added "distances" to the include list so we can calculate the score
                include=["documents", "metadatas", "distances", "embeddings"] 
            )
            
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                # ✅ FIX 2: Added the missing comma after 'i' and fixed 'metadeta' spelling
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    
                    # ChromaDB uses Cosine Distance by default (0 is perfect match, 1 is completely different)
                    # Converting distance to a Similarity Score (1 is perfect, 0 is different)
                    # ✅ THE FIX: Convert ChromaDB's default Squared L2 distance to Cosine Similarity
                    similarity_score = 1.0 - (distance / 2.0)
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'content': document,
                            'metadata': metadata,
                            'score': similarity_score,
                            'id': doc_id,
                            'rank': i + 1,
                            'distance': distance
                        })
                        # Moved this print statement outside the append block so it doesn't print for every single document
                    else:
                        print(f"Document '{doc_id}' filtered out due to low similarity score: {similarity_score:.4f}")

            print(f"Total retrieved: {len(retrieved_docs)} documents after applying score threshold.")
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            raise
rag_retrieval = Retrieval(vectorstore, embedding_manager)
rag_retrieval.retrieve("What is the content of the document?", score_threshold=0.5, top_k=3)

Retrieving documents for query: 'What is the content of the document?' with score threshold: 0.5 and top_k: 3
Generated embeddings for input: ['What is the content of the document?']
Embedding shape: (1, 384)
Query embedding shape: (384,)
Total retrieved: 3 documents after applying score threshold.


[{'content': '},\n  {\n    "id": 40,\n    "category": "Tech Discussion",\n    "tone": "Technical",\n    "content": "When studying social networks, calculating eigenvector centrality is computationally expensive for large graphs. What approximation algorithms do you prefer for large-scale data?"\n  },\n  {\n    "id": 41,\n    "category": "Community Discussion",\n    "tone": "Welcoming",\n    "content": "Welcome to all the new members joining MYpostmate today! Check out the #introductions channel and make sure to read the community guidelines regarding AI content generation."\n  },\n  {\n    "id": 42,\n    "category": "Tech Discussion",\n    "tone": "Practical",\n    "content": "If you want to understand how HTTP requests really work, try building a basic web server from scratch in Java using only the ServerSocket class. It strips away all the magic."\n  },\n  {\n    "id": 43,\n    "category": "Career & Job Prep",\n    "tone": "Curious",',
  'metadata': {'doc_index': 13,
   'content_leng

In [13]:
rag_retrieval

In [14]:
print(f"Total items in collection: {rag_retrieval.vectorstore.collection.count()}")



Total items in collection: 36


In [15]:
rag_retrieval.retrieve("What is the content of the document?", score_threshold=0.5, top_k=3)    

Retrieving documents for query: 'What is the content of the document?' with score threshold: 0.5 and top_k: 3
Generated embeddings for input: ['What is the content of the document?']
Embedding shape: (1, 384)
Query embedding shape: (384,)
Total retrieved: 3 documents after applying score threshold.


[{'content': '},\n  {\n    "id": 40,\n    "category": "Tech Discussion",\n    "tone": "Technical",\n    "content": "When studying social networks, calculating eigenvector centrality is computationally expensive for large graphs. What approximation algorithms do you prefer for large-scale data?"\n  },\n  {\n    "id": 41,\n    "category": "Community Discussion",\n    "tone": "Welcoming",\n    "content": "Welcome to all the new members joining MYpostmate today! Check out the #introductions channel and make sure to read the community guidelines regarding AI content generation."\n  },\n  {\n    "id": 42,\n    "category": "Tech Discussion",\n    "tone": "Practical",\n    "content": "If you want to understand how HTTP requests really work, try building a basic web server from scratch in Java using only the ServerSocket class. It strips away all the magic."\n  },\n  {\n    "id": 43,\n    "category": "Career & Job Prep",\n    "tone": "Curious",',
  'metadata': {'source': '..\\RAG-folder\\top-po

### Evaluation

In [20]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time

# 1 = harmful, 0 = safe

y_true = [1, 0, 1, 1, 0, 0, 1, 0, 1, 0]   # actual labels
y_pred = [1, 0, 1, 0, 0, 0, 1, 1, 1, 0]   # model predictions

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("Moderation Evaluation Metrics:")
print(f"Accuracy:  {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall:    {recall:.2f}")
print(f"F1 Score:  {f1:.2f}")

Moderation Evaluation Metrics:
Accuracy:  0.80
Precision: 0.80
Recall:    0.80
F1 Score:  0.80


In [21]:
# Ground truth: correct chunk index
ground_truth = [2, 0, 1, 3, 2]

# Retrieved chunks (Top-3 results)
retrieved = [
    [2, 4, 5],
    [1, 0, 3],
    [1, 2, 3],
    [0, 2, 3],
    [4, 2, 1]
]

In [22]:
def hit_at_k(ground_truth, retrieved, k=3):
    hits = 0
    for gt, ret in zip(ground_truth, retrieved):
        if gt in ret[:k]:
            hits += 1
    return hits / len(ground_truth)

hit_k = hit_at_k(ground_truth, retrieved, k=3)

print(f"Hit@3: {hit_k:.2f}")

Hit@3: 1.00


In [23]:
# Simulated relevance (1 = relevant, 0 = not)
relevance_scores = [
    [1, 0, 0],
    [0, 1, 0],
    [1, 1, 0],
    [0, 0, 1],
    [0, 1, 1]
]

relevant = sum(sum(r) for r in relevance_scores)
total = len(relevance_scores) * 3

context_relevance = relevant / total

print(f"Context Relevance: {context_relevance:.2f}")

Context Relevance: 0.47


In [24]:
def simulate_rag():
    time.sleep(0.2)  # simulate processing delay
    return "response"

start = time.time()
simulate_rag()
end = time.time()

latency = (end - start) * 1000  # ms

print(f"Latency: {latency:.2f} ms")

Latency: 201.19 ms


In [25]:
results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "Hit@3", "Context Relevance", "Latency (ms)"],
    "Value": [accuracy, precision, recall, f1, hit_k, context_relevance, latency]
})

results

,Metric,Value
0,Accuracy,0.800000
1,Precision,0.800000
2,Recall,0.800000
3,F1 Score,0.800000
4,Hit@3,1.000000
5,Context Relevance,0.466667
6,Latency (ms),201.187611


In [18]:
import sys
print(sys.executable)

c:\Users\Dell\postmate\MYpostman-APP\Backend\RAG-local\.venv\Scripts\python.exe


## Augmentation

### RAG retrieval pipeline with context and prompt to feed LLM